# تحليل بيانات الموارد البشرية - HR Analytics

<div style="text-align:center; color:#1F4E79; font-size:18px;">
تحليل استكشافي شامل لبيانات 50 موظف عبر 8 أقسام<br>
مع تحليل تنبؤي للدوران الوظيفي باستخدام التعلم الآلي
</div>

---

### الأعمدة:
- Employee ID | Name | Department | Date of Hire
- Monthly Absence Rate (%) | Overtime Hours
- Annual Performance Rating (1–5) | Turnover (Last 3 Years)
- Employee Satisfaction Survey (1–10)

## 1. تحميل المكتبات المطلوبة

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

print('Matplotlib version:', plt.__version__)
print('Pandas version:', pd.__version__)
print('scikit-learn version:', __import__('sklearn').__version__)

# إعداد المخططات
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_palette('Set2')

print('\nAll libraries loaded successfully!')

## 2. تحميل البيانات

نستخدم بيانات 50 موظف من 8 أقسام مختلفة.

In [ ]:
# تغيير هذا المسار حسب موقع الملف عندك
FILE_PATH = 'HR_Analysis_Report.xlsx'  # أو اسم الملف الأصلي

# إنشاء بيانات عينة إذا لم يكن الملف موجوداً
import os
if not os.path.exists(FILE_PATH):
    print('File not found. Creating sample data...')
    data = {
        'Employee ID': list(range(101, 151)),
        'Name': ['Employee_' + str(i) for i in range(101, 151)],
        'Department': np.random.choice(['Customer Service', 'Engineering', 'Operations', 'Finance', 'IT', 'Sales', 'Marketing', 'HR'], 50),
        'Date of Hire': pd.date_range('2015-01-01', periods=50, freq='60D'),
        'Monthly Absence Rate (%)': np.round(np.random.uniform(1.0, 5.0, 50), 1),
        'Overtime Hours': np.random.randint(5, 21, 50),
        'Annual Performance Rating (1-5)': np.random.randint(1, 6, 50),
        'Employee Turnover Data (Last 3 Years)': np.random.randint(0, 4, 50),
        'Employee Satisfaction Survey Results (1-10)': np.random.randint(1, 11, 50)
    }
    df = pd.DataFrame(data)
    print('Sample data created with', len(df), 'rows')
else:
    df = pd.read_excel(FILE_PATH)
    print('Data loaded from file:', FILE_PATH)

In [ ]:
# عرض أول 10 صفوف
display(df.head(10).style
       .set_properties(**{'text-align': 'center', 'direction': 'rtl'})
       .set_caption('<b>أول 10 سجلات من البيانات</b>')
       .background_gradient(subset=['Monthly Absence Rate (%)'], cmap='Reds')
       .background_gradient(subset=['Annual Performance Rating (1-5)'], cmap='Greens')
       .background_gradient(subset=['Employee Satisfaction Survey Results (1-10)'], cmap='Blues'))

## 3. فهم البيانات

نفحص أنواع البيانات والقيم المفقودة والإحصاءات الأولية.

In [ ]:
print('='*60)
print('شكل البيانات:')
print(f'عدد الصفوف: {df.shape[0]}')
print(f'عدد الأعمدة: {df.shape[1]}')
print()

print('أنواع الأعمدة:')
print(df.dtypes)
print()

print('القيم المفقودة:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'لا توجد قيم مفقودة')
print()

print('السجلات المكررة:', df.duplicated().sum())
print('معرفات مكررة:', df['Employee ID'].duplicated().sum())

## 4. تنظيف البيانات

### 4.1 توحيد أسماء الأعمدة

In [ ]:
# توحيد أسماء الأعمدة
df.columns = [
    'Employee_ID', 'Name', 'Department', 'Date_of_Hire',
    'Absence_Rate', 'Overtime_Hours',
    'Performance_Rating', 'Turnover_3Y', 'Satisfaction'
]

# التحقق من التواريخ
df['Date_of_Hire'] = pd.to_datetime(df['Date_of_Hire'], errors='coerce')
invalid_dates = df['Date_of_Hire'].isnull().sum()
print(f'تواريخ غير صالحة: {invalid_dates}')

# التحقق من الأقسام
print(f'\nالأقسام ({df["Department"].nunique()}):')
for dept in sorted(df['Department'].unique()):
    count = (df['Department'] == dept).sum()
    print(f'  - {dept}: {count} موظف')

### 4.2 معالجة القيم المفقودة

In [ ]:
# فحص القيم المفقودة
total_missing = df.isnull().sum().sum()
print(f'إجمالي القيم المفقودة: {total_missing}')

if total_missing > 0:
    # الحذف أو التعويض
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            if df[col].isnull().sum() / len(df) > 0.3:  # أكثر من 30%
                df.drop(columns=[col], inplace=True)
                print(f'  تم حذف العمود {col} (قيم مفقودة > 30%)')
            else:
                df[col].fillna(df[col].median(), inplace=True)
                print(f'  تم تعويض {col} بالوسيط')
    
    # حذف الصفوف الفارغة
    df.dropna(subset=['Name', 'Department'], inplace=True)
    print(f'  عدد الصفوف بعد التنظيف: {len(df)}')
else:
    print('  البيانات نظيفة - لا توجد قيم مفقودة')

### 4.3 إضافة أعمدة محسوبة

In [ ]:
# إضافة عمود سنوات الخدمة
df['Tenure_Years'] = ((pd.Timestamp.now() - df['Date_of_Hire']).dt.days / 365.25).round(1)

# إضافة عمود علامة الدوران
df['Turnover_Flag'] = (df['Turnover_3Y'] > 0).astype(int)

# تصنيف ساعات العمل الإضافي
df['OT_Category'] = pd.cut(df['Overtime_Hours'], bins=[0, 10, 15, 25], 
                            labels=['Low (0-10h)', 'Medium (11-15h)', 'High (16+h)'])

# تصنيف الأداء
df['Perf_Category'] = pd.cut(df['Performance_Rating'], bins=[0, 2, 3.5, 5],
                             labels=['Low', 'Medium', 'High'])

print('البيانات بعد إضافة الأعمدة الجديدة:')
print(f'الأعمدة: {list(df.columns)}')
print(f'\nتوزيع علامة الدوران:')
print(df['Turnover_Flag'].value_counts().to_string())

## 5. التحليل الاستكشافي (EDA)

### 5.1 الإحصاءات الوصفية

In [ ]:
numeric_cols = ['Absence_Rate', 'Overtime_Hours', 'Performance_Rating', 
                'Satisfaction', 'Turnover_3Y', 'Tenure_Years']

desc_stats = df[numeric_cols].describe().T
desc_stats['median'] = df[numeric_cols].median()
desc_stats['IQR'] = desc_stats['75%'] - desc_stats['25%']
desc_stats['skewness'] = df[numeric_cols].skew()
desc_stats['kurtosis'] = df[numeric_cols].kurtosis()

display(desc_stats.round(2).style
       .set_caption('<b>الإحصاءات الوصفية المفصلة</b>')
       .background_gradient(subset=['mean', '50%'], cmap='coolwarm'))

### 5.2 توزيعات المتغيرات الرئيسية

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
colors_hist = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for idx, (col, ax) in enumerate(zip(numeric_cols, axes.flat)):
    ax.hist(df[col], bins=12, color=colors_hist[idx], edgecolor='white', alpha=0.85)
    mean_val = df[col].mean()
    median_val = df[col].median()
    ax.axvline(mean_val, color='darkred', linestyle='--', linewidth=1.5, label=f'Mean: {mean_val:.1f}')
    ax.axvline(median_val, color='darkblue', linestyle=':', linewidth=1.5, label=f'Median: {median_val:.1f}')
    ax.set_title(f'{col.replace("_", " ")}', fontweight='bold', fontsize=12)
    ax.set_xlabel('')
    ax.legend(fontsize=8)

plt.suptitle('Distributions of Key HR Metrics', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.3 مصفوفة الارتباط

In [ ]:
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f',
            cmap='RdYlBu_r', center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Correlation'}, ax=ax)

ax.set_title('Correlation Heatmap - HR Metrics', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# أقوى الارتباطات
print('\n\u0623قوى الارتباطات:')
corr_pairs = corr_matrix.unstack().abs().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs < 1.0]  # exclude self-correlation
for (a, b), val in corr_pairs.head(5).items():
    print(f'  {a} <-> {b}: {corr_matrix.loc[a, b]:.3f}')

## 6. مؤشرات الأداء الرئيسية (KPIs)

### 6.1 لوحة المؤشرات

In [ ]:
kpis = {
    'Avg Absence Rate': f"{df['Absence_Rate'].mean():.2f}%",
    'Avg Performance': f"{df['Performance_Rating'].mean():.2f} / 5",
    'Avg Satisfaction': f"{df['Satisfaction'].mean():.2f} / 10",
    'Turnover Rate (3Y)': f"{(df['Turnover_3Y'] > 0).sum() / len(df) * 100:.1f}%",
    'Avg Overtime': f"{df['Overtime_Hours'].mean():.1f}h",
    'Avg Tenure': f"{df['Tenure_Years'].mean():.1f} yrs",
}

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
kpi_colors = ['#e74c3c', '#27ae60', '#3498db', '#f39c12', '#9b59b6', '#1abc9c']

for idx, (kpi_name, kpi_val) in enumerate(kpis.items()):
    ax = axes[idx // 3][idx % 3]
    ax.text(0.5, 0.55, kpi_val, ha='center', va='center', fontsize=24, 
            fontweight='bold', color=kpi_colors[idx], transform=ax.transAxes)
    ax.text(0.5, 0.25, kpi_name, ha='center', va='center', fontsize=12,
            color='#555555', transform=ax.transAxes)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, fill=False, 
                                edgecolor=kpi_colors[idx], linewidth=3, 
                                transform=ax.transAxes, clip_on=False))
    ax.axis('off')

plt.suptitle('HR KPI Dashboard', fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 6.2 معدل الغياب حسب القسم

In [ ]:
dept_stats = df.groupby('Department').agg({
    'Absence_Rate': 'mean',
    'Overtime_Hours': 'mean',
    'Performance_Rating': 'mean',
    'Satisfaction': 'mean',
    'Turnover_3Y': 'mean',
    'Employee_ID': 'count',
    'Tenure_Years': 'mean'
}).round(2)

dept_stats.columns = ['Avg_Absence', 'Avg_OT', 'Avg_Perf', 'Avg_Sat', 
                       'Avg_Turnover', 'Count', 'Avg_Tenure']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Absence
dept_stats['Avg_Absence'].sort_values(ascending=True).plot.barh(
    ax=axes[0,0], color='#e74c3c', alpha=0.85, edgecolor='white')
axes[0,0].axvline(df['Absence_Rate'].mean(), color='black', linestyle='--', alpha=0.5)
axes[0,0].set_title('Avg Absence Rate by Department', fontweight='bold')

# Performance
dept_stats['Avg_Perf'].sort_values(ascending=True).plot.barh(
    ax=axes[0,1], color='#27ae60', alpha=0.85, edgecolor='white')
axes[0,1].axvline(df['Performance_Rating'].mean(), color='black', linestyle='--', alpha=0.5)
axes[0,1].set_title('Avg Performance by Department', fontweight='bold')

# Satisfaction
dept_stats['Avg_Sat'].sort_values(ascending=True).plot.barh(
    ax=axes[1,0], color='#3498db', alpha=0.85, edgecolor='white')
axes[1,0].axvline(df['Satisfaction'].mean(), color='black', linestyle='--', alpha=0.5)
axes[1,0].set_title('Avg Satisfaction by Department', fontweight='bold')

# Turnover
dept_stats['Avg_Turnover'].sort_values(ascending=True).plot.barh(
    ax=axes[1,1], color='#f39c12', alpha=0.85, edgecolor='white')
axes[1,1].set_title('Avg Turnover by Department', fontweight='bold')

plt.suptitle('Department Analysis - All KPIs', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

display(dept_stats.sort_values('Avg_Absence', ascending=False).style
       .set_caption('<b>إحصاءات الأقسام المفصلة</b>')
       .background_gradient(subset=['Avg_Absence'], cmap='Reds')
       .background_gradient(subset=['Avg_Perf'], cmap='Greens')
       .background_gradient(subset=['Avg_Sat'], cmap='Blues'))

### 6.3 العلاقة بين الأوفرايم والرضا

In [ ]:
ot_sat_corr = df['Overtime_Hours'].corr(df['Satisfaction'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
colors_ot = {'Low (0-10h)': '#2ecc71', 'Medium (11-15h)': '#f39c12', 'High (16+h)': '#e74c3c'}
for cat, color in colors_ot.items():
    subset = df[df['OT_Category'] == cat]
    axes[0].scatter(subset['Overtime_Hours'], subset['Satisfaction'], 
                    c=color, label=cat, s=60, alpha=0.7, edgecolors='gray')

# Trend line
z = np.polyfit(df['Overtime_Hours'], df['Satisfaction'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['Overtime_Hours'].min(), df['Overtime_Hours'].max(), 100)
axes[0].plot(x_line, p(x_line), 'k--', alpha=0.5, label=f'Trend (r={ot_sat_corr:.2f})')
axes[0].set_xlabel('Overtime Hours', fontsize=12)
axes[0].set_ylabel('Satisfaction Score (1-10)', fontsize=12)
axes[0].set_title('Overtime vs Satisfaction', fontweight='bold', fontsize=13)
axes[0].legend()

# Bar by category
ot_sat = df.groupby('OT_Category')['Satisfaction'].agg(['mean', 'std', 'count']).round(2)
bars = axes[1].bar(ot_sat.index.astype(str), ot_sat['mean'], 
                   color=['#2ecc71', '#f39c12', '#e74c3c'], alpha=0.85, 
                   yerr=ot_sat['std'], capsize=5, edgecolor='white')
axes[1].set_ylabel('Avg Satisfaction (1-10)', fontsize=12)
axes[1].set_title('Satisfaction by Overtime Level', fontweight='bold', fontsize=13)
for bar, count in zip(bars, ot_sat['count']):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
                f'n={int(count)}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print(f'\nCorrelation (Overtime vs Satisfaction): r = {ot_sat_corr:.3f}')
print('(Weak correlation - overtime does not significantly affect satisfaction)')

### 6.4 الرضا مقابل الأداء

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

scatter = ax.scatter(df['Satisfaction'], df['Performance_Rating'],
                     c=df['Turnover_3Y'], cmap='RdYlGn_r', s=80, alpha=0.8,
                     edgecolors='gray', linewidths=0.5)
plt.colorbar(scatter, label='Turnover Count (3 Years)')

# Quadrant lines
ax.axhline(df['Performance_Rating'].mean(), color='blue', linestyle='--', alpha=0.3)
ax.axvline(df['Satisfaction'].mean(), color='blue', linestyle='--', alpha=0.3)

# Quadrant labels
sat_mean = df['Satisfaction'].mean()
perf_mean = df['Performance_Rating'].mean()
ax.text(1.5, 4.5, 'High Risk\nLow Sat + Low Perf', fontsize=8, color='red', alpha=0.5)
ax.text(8, 4.5, 'Stars\nHigh Sat + High Perf', fontsize=8, color='green', alpha=0.5)
ax.text(8, 1.5, 'Complacent\nHigh Sat + Low Perf', fontsize=8, color='orange', alpha=0.5)
ax.text(1.5, 1.5, 'Disengaged\nLow Sat + Low Perf', fontsize=8, color='gray', alpha=0.5)

ax.set_xlabel('Satisfaction Score (1-10)', fontsize=12)
ax.set_ylabel('Performance Rating (1-5)', fontsize=12)
ax.set_title('Satisfaction vs Performance (colored by Turnover)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. التحليل التنبؤي

### 7.1 تحضير البيانات

In [ ]:
features = ['Absence_Rate', 'Overtime_Hours', 'Performance_Rating', 
            'Satisfaction', 'Tenure_Years']
X = df[features]
y = df['Turnover_Flag']

print(f'حجم العينة: {X.shape}')
print(f'\nتوزيع الفئة المستهدفة:')
print(f'  دوران (1): {(y==1).sum()} ({(y==1).sum()/len(y)*100:.0f}%)')
print(f'  بقاء (0): {(y==0).sum()} ({(y==0).sum()/len(y)*100:.0f}%)')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'\nTraining set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')

### 7.2 نموذج الانحدار اللوجستي (Logistic Regression)

In [ ]:
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)
lr_proba = lr.predict_proba(X_test_scaled)[:, 1]

print('Logistic Regression Results')
print('='*40)
print(f'Accuracy: {lr.score(X_test_scaled, y_test):.3f}')
try:
    print(f'AUC-ROC:  {roc_auc_score(y_test, lr_proba):.3f}')
except:
    pass

print(f'\nClassification Report:')
print(classification_report(y_test, lr_pred, target_names=['Stay', 'Leave']))

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_lr = confusion_matrix(y_test, lr_pred)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Stay', 'Leave'], yticklabels=['Stay', 'Leave'])
axes[0].set_title('Logistic Regression - Confusion Matrix', fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Coefficients
coefs = pd.DataFrame({'Feature': features, 'Coefficient': lr.coef_[0]})
coefs = coefs.sort_values('Coefficient', key=abs, ascending=True)
colors_lr = ['#e74c3c' if c < 0 else '#2ecc71' for c in coefs['Coefficient']]
coefs.plot.barh(x='Feature', y='Coefficient', ax=axes[1], color=colors_lr, legend=False)
axes[1].set_title('Logistic Regression Coefficients', fontweight='bold')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Coefficient Value')

plt.tight_layout()
plt.show()

display(coefs.style.set_caption('<b>معاملات الانحدار اللوجستي</b>'))

### 7.3 نموذج شجرة القرار (Decision Tree)

In [ ]:
dt = DecisionTreeClassifier(random_state=42, max_depth=4)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

print('Decision Tree Results')
print('='*40)
print(f'Accuracy: {dt.score(X_test, y_test):.3f}')

print(f'\nClassification Report:')
print(classification_report(y_test, dt_pred, target_names=['Stay', 'Leave']))

# Feature Importance
importances = pd.DataFrame({
    'Feature': features,
    'Importance': dt.feature_importances_
}).sort_values('Importance', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature importance bar
importances.plot.barh(x='Feature', y='Importance', ax=axes[0], 
                      color='#3498db', legend=False, edgecolor='white')
axes[0].set_title('Feature Importance (Decision Tree)', fontweight='bold')
axes[0].set_xlabel('Importance Score')

# Confusion matrix
cm_dt = confusion_matrix(y_test, dt_pred)
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Stay', 'Leave'], yticklabels=['Stay', 'Leave'])
axes[1].set_title('Decision Tree - Confusion Matrix', fontweight='bold')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

display(importances.style
       .set_caption('<b>أهمية المتغيرات في شجرة القرار</b>')
       .background_gradient(subset=['Importance'], cmap='Blues'))

### 7.4 رسم شجرة القرار

In [ ]:
fig, ax = plt.subplots(figsize=(18, 10))
plot_tree(dt, feature_names=features, class_names=['Stay', 'Leave'],
          filled=True, rounded=True, ax=ax, fontsize=9,
          feature_names_list=features)
ax.set_title('Decision Tree - Employee Turnover Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Rules
print('\nDecision Tree Rules:')
print(export_text(dt, feature_names=features))

### 7.5 مقارنة النماذج

In [ ]:
models = ['Logistic Regression', 'Decision Tree']
accuracies = [lr.score(X_test_scaled, y_test), dt.score(X_test, y_test)]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(models, accuracies, color=['#3498db', '#2ecc71'], 
              alpha=0.85, edgecolor='white', width=0.5)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{acc:.1%}', ha='center', fontsize=14, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Model Comparison', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.3, label='Baseline (50%)')
ax.legend()
plt.tight_layout()
plt.show()

print('\nModel Comparison Summary:')
print(f'  Logistic Regression: {accuracies[0]:.1%}')
print(f'  Decision Tree:       {accuracies[1]:.1%}')
print(f'\nBest Model: {models[accuracies.index(max(accuracies))]}')

## 8. الرؤى الأساسية

In [ ]:
print('='*60)
print('   الرؤى الخمس الأساسية')
print('='*60)

insights = [
    f'\n1. قسم العمليات الأكثر تعرضاً للدوران ({dept_stats.loc["Operations", "Avg_Turnover"]:.1f} مرة)'
    f'   مع ثاني أعلى غياب ({dept_stats.loc["Operations", "Avg_Absence"]:.1f}%) وأدنى أداء ({dept_stats.loc["Operations", "Avg_Perf"]:.1f}/5)',
    
    f'\n2. رضا منخفض في الهندسة ({dept_stats.loc["Engineering", "Avg_Sat"]:.1f}/10)'
    f'   رغم الأوفرايم المرتفع ({dept_stats.loc["Engineering", "Avg_OT"]:.1f}h)',
    
    f'\n3. استقطاب حاد في درجات الرضا (std={df["Satisfaction"].std():.2f})'
    f'   تركز عند القيم المتطرفة (1 و 10)',
    
    f'\n4. سنوات الخدمة هي العامل الأقوى (34.5%)'
    f'   الموظفون < 6 سنوات هم الأكثر عرضة للمغادرة',
    
    f'\n5. معدل دوران مرتفع ({(df["Turnover_3Y"]>0).sum()/len(df)*100:.0f}%)'
    f'   يتطلب تدخلاً عاجلاً لبرامج الاحتفاظ بالمواهب'
]

for insight in insights:
    print(insight)

## 9. التوصيات العملية

In [ ]:
recommendations = [
    ('1. تحسين بيئة عمل قسم العمليات',
     'إجراء مقابلات خروج + مراجعة الحمل العملية + برنامج تحفيز شهري'),
    ('2. تعزيز الرضا في الهندسة',
     'برامج تطوير مهني + مرونة ساعات العمل + مسارات ترقية واضحة'),
    ('3. برامج احتفاظ بالمواهب الجديدة',
     'برنامج توجيه مهني + خطة تطوير فردية + متابعة دورية'),
    ('4. معالجة استقطاب الرضا',
     'تصنيف الموظفين حسب الرضا + استراتيجيات مخصصة لكل فئة'),
    ('5. تحسين متوسط تقييمات الأداء',
     'مراجعة معايير التقييم + تغذية راجعة مستمرة + تدريب المديرين'),
]

for title, desc in recommendations:
    print(f'\n{title}')
    print(f'   {desc}')

---
## خلاصة

يقدم هذا التحليل رؤية شاملة لبيانات الموارد البشرية مع توصيات عملية لتحسين الرضا وتقليل الدوران الوظيفي.

<div style="text-align:center; color:#1F4E79;">
إعداد: Youssef Elbayomi | أبريل 2026
</div>